In [ ]:
import datetime
import pandas as pd
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from os import listdir
from scipy.interpolate import RegularGridInterpolator
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm_notebook

In [ ]:
## 从NC文件时间转PD时间
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))

## 获取文件夹里所有NC文件
def getfiles(filed):
    files=listdir(filed)
    files.sort()
    files=files[0:]
    files_n=[filed+'/'+i for i in files]
    return files_n

## 计算等间距6H日平均
def getDaymen(test,times):
    data = test
    time_index = times

    # 将时间序列转化为日期序列，使用 groupby 方法对每天的数据进行分组，最后求平均值
    date_index = time_index.date
    grouped = pd.DataFrame(data.reshape(-1, test.shape[1]*test.shape[2])).groupby(date_index).mean()
    result = grouped.values.reshape(-1, test.shape[1],test.shape[2])
    return result

das1=nc.Dataset('/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/1993_150E_180.nc')
das2=nc.Dataset('/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/1993_0_70W.nc')
lon1=np.array(das1['longitude'])
lon2=np.array(das2['longitude'])
lat=np.array(das1['latitude'])
lon=np.concatenate([lon1[:-1],lon2+360])
print(das1.variables.keys())

In [ ]:
pd.to_datetime(list(map(cftime2pdtime,nc.num2date(das1['time'],das1['time'].units))))

In [ ]:
times=[]
for i  in tqdm_notebook(range(1993,2023)):
    das1=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_150E_180.nc")
    times.append(pd.to_datetime(list(map(cftime2pdtime,nc.num2date(das1['time'],das1['time'].units)))))
times=pd.to_datetime(np.concatenate(times, axis=0))

In [ ]:
mslhfs=[]
msnlwrfs=[]
msnswrfs=[]
msshfs=[]
for i  in tqdm_notebook(range(1993,2023)):
    das1=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_150E_180.nc")
    das2=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_0_70W.nc")
    if np.abs(np.sum(das1['mslhf'][:,:,-1]-das2['mslhf'][:,:,0]))>1000:
        print(i)
    mslhf=np.concatenate([das1['mslhf'],das2['mslhf'][:,:,1:]],axis=-1)
    mslhfs.append(mslhf)
    msnlwrf=np.concatenate([das1['msnlwrf'],das2['msnlwrf'][:,:,1:]],axis=-1)
    msnlwrfs.append(msnlwrf)
    msnswrf=np.concatenate([das1['msnswrf'],das2['msnswrf'][:,:,1:]],axis=-1)
    msnswrfs.append(msnswrf)
    msshf=np.concatenate([das1['msshf'],das2['msshf'][:,:,1:]],axis=-1)
    msshfs.append(msshf)
    
    

In [ ]:
LH=getDaymen(np.array(np.concatenate(mslhfs, axis=0)),times)
SH=getDaymen(np.array(np.concatenate(msshfs, axis=0)),times)
LW=getDaymen(np.array(np.concatenate(msnlwrfs, axis=0)),times)
SW=getDaymen(np.array(np.concatenate(msnswrfs, axis=0)),times)

In [ ]:
np.savez('ERAdata.npz',LH=LH,SH=SH,LW=LW,SW=SW,lon=lon,lat=lat)

In [ ]:
das1=nc.Dataset(rf"D:\OneDrive\heat_budget\U&V&SLP150_180.nc")
das2=nc.Dataset(rf"D:\OneDrive\heat_budget\U&V&SLP180_250.nc")
lon1=np.array(das1['longitude'])
lon2=np.array(das2['longitude'])
lat=np.array(das1['latitude'])
lon=np.concatenate([lon1[:-1],lon2+360])
lon

In [8]:
das1.variables.keys()

dict_keys(['longitude', 'latitude', 'time', 'u10', 'v10', 'msl'])

In [ ]:
u101=np.arra